In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/RuzgarTahmin/yeni', exist_ok=True)
print("Hazır")

Mounted at /content/drive
Hazır


In [2]:
%%writefile /content/drive/MyDrive/RuzgarTahmin/rt_core.py
import numpy as np, pandas as pd
from scipy import stats

STEP_MIN = 10
KAPASITE = 1500.0

BASE_COLS = ['Wspd','Wdir','Etmp','Itmp','Ndir','Pab1','Pab2','Pab3','Prtv',
             'T2m','Sp','RelH','Wspd_w','Wdir_w','Tp']
TIME_COLS = ['saat','dakika','haftanin_gunu','ay','yilin_gunu',
             'saat_sin','saat_cos','ay_sin','ay_cos','wdir_sin','wdir_cos']
LAGS = [6,12,18,24,36,72,144]
AR_COLS = (['Patv'] + [f'Patv_lag{l}' for l in LAGS] +
           ['Patv_rol6','Patv_rol24','Wspd_rol6'])
FEATURES = BASE_COLS + ['is_missing'] + TIME_COLS + AR_COLS

def load_turbine(df_all, turb_id):
    d = df_all[df_all['TurbID'] == turb_id].copy()
    d['Tmstamp'] = pd.to_datetime(d['Tmstamp'])
    d = d.sort_values('Tmstamp').drop_duplicates('Tmstamp').set_index('Tmstamp')
    full = pd.date_range(d.index.min(), d.index.max(), freq=f'{STEP_MIN}min')
    d = d.reindex(full)
    d.index.name = 'Tmstamp'
    return d

def add_features(d):
    d = d.copy()
    era5 = ['T2m','Sp','RelH','Wspd_w','Wdir_w','Tp']
    d[era5] = d[era5].interpolate(method='linear', limit=6)
    d['is_missing'] = d['Patv'].isna().astype(int)
    d['Patv'] = d['Patv'].clip(lower=0)
    idx = d.index
    d['saat'] = idx.hour; d['dakika'] = idx.minute
    d['haftanin_gunu'] = idx.dayofweek; d['ay'] = idx.month; d['yilin_gunu'] = idx.dayofyear
    d['saat_sin'] = np.sin(2*np.pi*d['saat']/24); d['saat_cos'] = np.cos(2*np.pi*d['saat']/24)
    d['ay_sin']   = np.sin(2*np.pi*d['ay']/12);   d['ay_cos']   = np.cos(2*np.pi*d['ay']/12)
    d['wdir_sin'] = np.sin(np.deg2rad(d['Wdir'])); d['wdir_cos'] = np.cos(np.deg2rad(d['Wdir']))
    for l in LAGS: d[f'Patv_lag{l}'] = d['Patv'].shift(l)
    d['Patv_rol6']  = d['Patv'].shift(1).rolling(6).mean()
    d['Patv_rol24'] = d['Patv'].shift(1).rolling(24).mean()
    d['Wspd_rol6']  = d['Wspd'].shift(1).rolling(6).mean()
    return d

def chrono_split(d, frac=(0.70,0.15,0.15), purge=0):
    n = len(d); i1 = int(n*frac[0]); i2 = int(n*(frac[0]+frac[1]))
    tr = d.iloc[:i1]; va = d.iloc[i1+purge:i2]; te = d.iloc[i2+purge:]
    return tr, va, te

def make_windows(X, patv, seq_len, H):
    N, F = X.shape
    rowok = ~np.isnan(X).any(1)
    csum = np.concatenate([[0], np.cumsum(rowok)])
    t = np.arange(seq_len-1, N-H)
    win_ok = (csum[t+1] - csum[t+1-seq_len]) == seq_len
    tgt_ok = ~np.isnan(patv[t+H]) & ~np.isnan(patv[t])
    t = t[win_ok & tgt_ok]
    Xw = np.stack([X[ti-seq_len+1:ti+1] for ti in t]) if len(t) else np.empty((0,seq_len,F))
    return Xw.astype(np.float32), t, patv[t+H], patv[t]

def metrics(y, p, y_cur, cap=KAPASITE):
    mae = np.mean(np.abs(y-p)); rmse = np.sqrt(np.mean((y-p)**2))
    r2 = 1 - np.sum((y-p)**2)/np.sum((y-y.mean())**2)
    mae_pers = np.mean(np.abs(y-y_cur))
    return dict(MAE=mae, RMSE=rmse, nMAE=100*mae/cap, nRMSE=100*rmse/cap, R2=r2,
                Skill=100*(mae_pers-mae)/mae_pers)

def lsi(e_causal, e_leaky):
    return 100*(e_causal-e_leaky)/e_causal

def dm_test(y, p1, p2, h=1, loss='abs'):
    e1, e2 = y-p1, y-p2
    d = (np.abs(e1)-np.abs(e2)) if loss=='abs' else (e1**2-e2**2)
    T = len(d); dbar = d.mean(); dc = d-dbar
    lrv = np.dot(dc,dc)/T
    for k in range(1, max(h,1)):
        lrv += 2*(1-k/h)*np.dot(dc[k:], dc[:-k])/T
    lrv = max(lrv, 1e-12)
    dm = dbar/np.sqrt(lrv/T)
    dm_c = dm*np.sqrt((T+1-2*h+h*(h-1)/T)/T)
    p = 2*(1-stats.t.cdf(abs(dm_c), df=T-1))
    return dict(DM=dm_c, p_raw=p, T=T, loss=loss)

def holm(pvals):
    p = np.asarray(pvals, float); m = len(p)
    order = np.argsort(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m-rank)*p[i]); adj[i] = min(1.0, run)
    return adj

Overwriting /content/drive/MyDrive/RuzgarTahmin/rt_core.py


In [3]:
import sys, json, numpy as np, pandas as pd
sys.path.insert(0,'/content/drive/MyDrive/RuzgarTahmin')
import importlib, rt_core; importlib.reload(rt_core)
from rt_core import *
df_all = pd.read_csv('/content/drive/MyDrive/RuzgarTahmin/veri/sdwpf_2001_2112_full.csv')
print(df_all.shape, '| Türbin sayısı:', df_all.TurbID.nunique())
print('Özellik sayısı:', len(FEATURES))

(11361190, 18) | Türbin sayısı: 134
Özellik sayısı: 38


In [4]:
g = df_all.groupby('TurbID')
tab = pd.DataFrame({'patv_max': g['Patv'].max(),
                    'eksik_oran': g['Patv'].apply(lambda s: s.isna().mean()),
                    'ort_guc': g['Patv'].mean()})
print(tab.describe().round(2))
ranked = tab.sort_values('eksik_oran')
pos = [int(round(q*(len(ranked)-1))) for q in (0,.25,.5,.75,1)]
secilen = sorted(set(ranked.index[pos].tolist()+[57]))
print('\nSeçilen türbinler:', secilen)
print(tab.loc[secilen].round(3))

       patv_max  eksik_oran  ort_guc
count    134.00      134.00   134.00
mean    1532.46        0.04   350.92
std       17.66        0.01    35.70
min     1520.51        0.04   247.96
25%     1520.99        0.04   328.36
50%     1521.40        0.04   347.99
75%     1559.83        0.04   374.13
max     1561.38        0.11   435.75

Seçilen türbinler: [10, 57, 61, 83, 84, 120]
        patv_max  eksik_oran  ort_guc
TurbID                               
10       1560.65       0.038  412.359
57       1521.49       0.041  354.884
61       1520.97       0.043  350.821
83       1561.38       0.115  360.860
84       1520.89       0.045  342.781
120      1521.42       0.041  310.701


In [5]:
H_LIST = [1,6,12,18,24,36]
SEQ_LEN = 12; PURGE = 36
d = add_features(load_turbine(df_all, 57))
tr, va, te = chrono_split(d, purge=PURGE)
for ad, x in (('egitim',tr),('dogrulama',va),('test',te)):
    print(f"{ad:10s}: {x.index.min()} → {x.index.max()} | satır={len(x):,}")

mn = tr[FEATURES].min().values; mx = tr[FEATURES].max().values
rng_ = np.where(mx-mn==0, 1, mx-mn)
sc = lambda x: (x[FEATURES].values-mn)/rng_
rows = []
for H in H_LIST:
    r = {'H_adim': H, 'H_saat': round(H/6,2)}
    for ad, x in (('egitim',tr),('dogrulama',va),('test',te)):
        Xw, t, y, yc = make_windows(sc(x), x['Patv'].values, SEQ_LEN, H)
        r['n_'+ad] = len(t)
        if ad == 'test':
            r['persist_MAE'] = np.mean(np.abs(y-yc))
            r['persist_RMSE'] = np.sqrt(np.mean((y-yc)**2))
    rows.append(r)
pt = pd.DataFrame(rows)
print(pt.round(2).to_string(index=False))
print('Patv max (T57):', d['Patv'].max(), '| test is_missing oranı:', round(te['is_missing'].mean(),4))

egitim    : 2020-01-01 00:10:00 → 2021-05-26 16:50:00 | satır=73,685
dogrulama : 2021-05-26 23:00:00 → 2021-09-13 08:30:00 | satır=15,754
test      : 2021-09-13 14:40:00 → 2022-01-01 00:10:00 | satır=15,754
 H_adim  H_saat  n_egitim  n_dogrulama  n_test  persist_MAE  persist_RMSE
      1    0.17     17680        12569   12979        44.72         89.48
      6    1.00     17662        12552   12963       109.84        193.81
     12    2.00     17644        12535   12953       155.40        254.42
     18    3.00     17631        12523   12942       190.81        299.50
     24    4.00     17620        12509   12936       217.96        335.59
     36    6.00     17595        12486   12915       258.92        385.68
Patv max (T57): 1521.49 | test is_missing oranı: 0.0269


In [6]:
prot = {'turbin':57,'ozellikler':FEATURES,'seq_len':SEQ_LEN,'purge_adim':PURGE,
        'ufuklar_adim':H_LIST,'secilen_turbinler':[int(s) for s in secilen],
        'bolme':{ad:{'bas':str(x.index.min()),'bit':str(x.index.max()),'satir':len(x)}
                 for ad,x in (('egitim',tr),('dogrulama',va),('test',te))},
        'pencere_ve_persistence': pt.to_dict('records')}
json.dump(prot, open('/content/drive/MyDrive/RuzgarTahmin/yeni/protokol_t57.json','w'), indent=1, default=str)
print('✓ Kaydedildi')

✓ Kaydedildi


In [7]:
# TEŞHİS: boşluklar nerede?
orig_ts = pd.to_datetime(df_all.loc[df_all.TurbID==57,'Tmstamp'])
present = d.index.isin(orig_ts)
X_all = (d[FEATURES].values - mn)/rng_
rowok = ~np.isnan(X_all).any(1)

# 1) Aylık: eksik zaman damgası %, Patv NaN %, tamamen dolu satır %
m = pd.DataFrame({'ay': d.index.to_period('M'),
                  'zaman_eksik': ~present,
                  'patv_nan': present & d['Patv'].isna().values,
                  'satir_tam': rowok})
print(m.groupby('ay').mean().mul(100).round(1).to_string())

# 2) Eğitimde hangi özellik en çok NaN üretiyor?
nan_oran = tr[FEATURES].isna().mean().mul(100).sort_values(ascending=False)
print('\nEğitimde en çok NaN içeren özellikler (%):')
print(nan_oran.head(10).round(1))

         zaman_eksik  patv_nan  satir_tam
ay                                       
2020-01         66.7       5.6        0.0
2020-02         66.7       0.0        0.0
2020-03         66.7       1.7        0.0
2020-04        100.0       0.0        0.0
2020-05         66.7       0.1        0.0
2020-06         66.7       0.0        0.0
2020-07         66.7       1.8        0.0
2020-08         66.7       0.0        0.0
2020-09         66.7       0.2        0.0
2020-10         66.7       0.0        0.0
2020-11         66.7       0.3        0.0
2020-12         67.0       0.0        0.0
2021-01          0.0      22.7       69.2
2021-02          0.0       0.2       95.1
2021-03          0.0       0.2       96.0
2021-04          0.0       0.1       96.8
2021-05          0.0       0.4       93.9
2021-06          0.0       0.2       95.9
2021-07          0.0      25.7       68.6
2021-08          0.0       0.5       92.2
2021-09          0.0       1.9       90.4
2021-10          0.0       0.1    

In [8]:
raw = df_all.loc[df_all.TurbID==57, ['Tmstamp','Patv']].copy()
raw['Tmstamp'] = pd.to_datetime(raw['Tmstamp'])
print('Toplam satır:', len(raw), '| tekrar eden zaman damgası:', raw.Tmstamp.duplicated().sum())
raw = raw.drop_duplicates('Tmstamp').sort_values('Tmstamp')
dt = raw.Tmstamp.diff().dt.total_seconds()/60
for yil in (2020, 2021):
    print(f'\n{yil}: satır={ (raw.Tmstamp.dt.year==yil).sum() }  | adım aralığı (dk):')
    print(dt[raw.Tmstamp.dt.year==yil].value_counts().head(5))

Toplam satır: 84785 | tekrar eden zaman damgası: 0

2020: satır=32224  | adım aralığı (dk):
Tmstamp
15.0       32222
43215.0        1
Name: count, dtype: int64

2021: satır=52559  | adım aralığı (dk):
Tmstamp
10.0     52558
495.0        1
Name: count, dtype: int64


In [9]:
%%writefile /content/drive/MyDrive/RuzgarTahmin/rt_core.py
import numpy as np, pandas as pd
from scipy import stats

STEP_MIN = 30
KAPASITE = 1500.0

BASE_COLS = ['Wspd','Wdir','Etmp','Itmp','Ndir','Pab1','Pab2','Pab3','Prtv',
             'T2m','Sp','RelH','Wspd_w','Wdir_w','Tp']
TIME_COLS = ['saat','haftanin_gunu','ay','yilin_gunu',
             'saat_sin','saat_cos','ay_sin','ay_cos','wdir_sin','wdir_cos']
LAG_H = [1,2,3,4,6,12,24]
LAGS = [2*h for h in LAG_H]
AR_COLS = (['Patv'] + [f'Patv_lag{h}h' for h in LAG_H] +
           ['Patv_rol1h','Patv_rol4h','Wspd_rol1h'])
FEATURES = BASE_COLS + ['is_missing'] + TIME_COLS + AR_COLS

def load_turbine(df_all, turb_id):
    d = df_all[df_all['TurbID'] == turb_id].copy()
    d['Tmstamp'] = pd.to_datetime(d['Tmstamp'])
    d = d.sort_values('Tmstamp').drop_duplicates('Tmstamp').set_index('Tmstamp')
    d = d[d.index.minute.isin([10, 40])]
    full = pd.date_range(d.index.min(), d.index.max(), freq=f'{STEP_MIN}min')
    d = d.reindex(full); d.index.name = 'Tmstamp'
    return d

def add_features(d):
    d = d.copy()
    era5 = ['T2m','Sp','RelH','Wspd_w','Wdir_w','Tp']
    d[era5] = d[era5].interpolate(method='linear', limit=2)
    d['is_missing'] = d['Patv'].isna().astype(int)
    d['Patv'] = d['Patv'].clip(lower=0)
    idx = d.index
    d['saat'] = idx.hour; d['haftanin_gunu'] = idx.dayofweek
    d['ay'] = idx.month; d['yilin_gunu'] = idx.dayofyear
    d['saat_sin'] = np.sin(2*np.pi*d['saat']/24); d['saat_cos'] = np.cos(2*np.pi*d['saat']/24)
    d['ay_sin']   = np.sin(2*np.pi*d['ay']/12);   d['ay_cos']   = np.cos(2*np.pi*d['ay']/12)
    d['wdir_sin'] = np.sin(np.deg2rad(d['Wdir'])); d['wdir_cos'] = np.cos(np.deg2rad(d['Wdir']))
    for h, l in zip(LAG_H, LAGS): d[f'Patv_lag{h}h'] = d['Patv'].shift(l)
    d['Patv_rol1h'] = d['Patv'].shift(1).rolling(2).mean()
    d['Patv_rol4h'] = d['Patv'].shift(1).rolling(8).mean()
    d['Wspd_rol1h'] = d['Wspd'].shift(1).rolling(2).mean()
    return d

def chrono_split(d, frac=(0.70,0.15,0.15), purge=0):
    n = len(d); i1 = int(n*frac[0]); i2 = int(n*(frac[0]+frac[1]))
    return d.iloc[:i1], d.iloc[i1+purge:i2], d.iloc[i2+purge:]

def make_windows(X, patv, seq_len, H):
    N, F = X.shape
    rowok = ~np.isnan(X).any(1)
    csum = np.concatenate([[0], np.cumsum(rowok)])
    t = np.arange(seq_len-1, N-H)
    win_ok = (csum[t+1] - csum[t+1-seq_len]) == seq_len
    tgt_ok = ~np.isnan(patv[t+H]) & ~np.isnan(patv[t])
    t = t[win_ok & tgt_ok]
    Xw = np.stack([X[ti-seq_len+1:ti+1] for ti in t]) if len(t) else np.empty((0,seq_len,F))
    return Xw.astype(np.float32), t, patv[t+H], patv[t]

def metrics(y, p, y_cur, cap=KAPASITE):
    mae = np.mean(np.abs(y-p)); rmse = np.sqrt(np.mean((y-p)**2))
    r2 = 1 - np.sum((y-p)**2)/np.sum((y-y.mean())**2)
    mae_pers = np.mean(np.abs(y-y_cur))
    return dict(MAE=mae, RMSE=rmse, nMAE=100*mae/cap, nRMSE=100*rmse/cap, R2=r2,
                Skill=100*(mae_pers-mae)/mae_pers)

def lsi(e_causal, e_leaky):
    return 100*(e_causal-e_leaky)/e_causal

def dm_test(y, p1, p2, h=1, loss='abs'):
    e1, e2 = y-p1, y-p2
    d = (np.abs(e1)-np.abs(e2)) if loss=='abs' else (e1**2-e2**2)
    T = len(d); dbar = d.mean(); dc = d-dbar
    lrv = np.dot(dc,dc)/T
    for k in range(1, max(h,1)):
        lrv += 2*(1-k/h)*np.dot(dc[k:], dc[:-k])/T
    lrv = max(lrv, 1e-12)
    dm = dbar/np.sqrt(lrv/T)
    dm_c = dm*np.sqrt((T+1-2*h+h*(h-1)/T)/T)
    p = 2*(1-stats.t.cdf(abs(dm_c), df=T-1))
    return dict(DM=dm_c, p_raw=p, T=T, loss=loss)

def holm(pvals):
    p = np.asarray(pvals, float); m = len(p)
    order = np.argsort(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m-rank)*p[i]); adj[i] = min(1.0, run)
    return adj

Overwriting /content/drive/MyDrive/RuzgarTahmin/rt_core.py


In [10]:
import sys, importlib, json, numpy as np, pandas as pd
sys.path.insert(0,'/content/drive/MyDrive/RuzgarTahmin')
import rt_core; importlib.reload(rt_core)
from rt_core import *
print('Özellik sayısı:', len(FEATURES), '| adım:', STEP_MIN, 'dk')

Özellik sayısı: 37 | adım: 30 dk


In [11]:
# ufuklar 30 dk adımıyla: 1=30dk, 2=1sa, 4=2sa, 6=3sa, 8=4sa, 12=6sa
H_LIST = [1,2,4,6,8,12]
SEQ_OPTS = [4,8,12]          # geçmiş pencere: 2sa, 4sa, 6sa (doğrulamada seçilecek)
PURGE = 12                   # 6 saat boşluk

d = add_features(load_turbine(df_all, 57))
print('Izgara adımı (30dk) kontrolü:', d.index.to_series().diff().dropna().value_counts().to_dict())
tr, va, te = chrono_split(d, purge=PURGE)
for ad, x in (('egitim',tr),('dogrulama',va),('test',te)):
    print(f"{ad:10s}: {x.index.min()} → {x.index.max()} | satır={len(x):,}")

mn = tr[FEATURES].min().values; mx = tr[FEATURES].max().values
rng_ = np.where(mx-mn==0, 1, mx-mn)
sc = lambda x: (x[FEATURES].values-mn)/rng_

rows = []
for L in SEQ_OPTS:
    for H in H_LIST:
        r = {'L_adim':L, 'L_saat':L/2, 'H_adim':H, 'H_saat':H/2}
        for ad, x in (('egitim',tr),('dogrulama',va),('test',te)):
            Xw, t, y, yc = make_windows(sc(x), x['Patv'].values, L, H)
            r['n_'+ad] = len(t)
            if ad == 'test':
                r['persist_MAE'] = np.mean(np.abs(y-yc))
        rows.append(r)
pt = pd.DataFrame(rows)
print(pt.round(2).to_string(index=False))
print('Test is_missing oranı:', round(te['is_missing'].mean(),4))

Izgara adımı (30dk) kontrolü: {Timedelta('0 days 00:30:00'): 35088}
egitim    : 2020-01-01 00:10:00 → 2021-05-26 16:40:00 | satır=24,562
dogrulama : 2021-05-26 23:10:00 → 2021-09-13 08:10:00 | satır=5,251
test      : 2021-09-13 14:40:00 → 2022-01-01 00:10:00 | satır=5,252
 L_adim  L_saat  H_adim  H_saat  n_egitim  n_dogrulama  n_test  persist_MAE
      4     2.0       1     0.5     21196         4705    4885        77.80
      4     2.0       2     1.0     21182         4702    4881       108.41
      4     2.0       4     2.0     21158         4697    4873       153.78
      4     2.0       6     3.0     21137         4693    4865       187.89
      4     2.0       8     4.0     21117         4688    4859       214.98
      4     2.0      12     6.0     21083         4680    4845       255.53
      8     4.0       1     0.5     20907         4656    4833        78.24
      8     4.0       2     1.0     20895         4653    4829       108.79
      8     4.0       4     2.0     20873  

In [12]:
prot = {'turbin':57,'adim_dk':STEP_MIN,'ozellikler':FEATURES,'seq_secenekleri':SEQ_OPTS,
        'purge_adim':PURGE,'ufuklar_adim':H_LIST,
        'bolme':{ad:{'bas':str(x.index.min()),'bit':str(x.index.max()),'satir':len(x)}
                 for ad,x in (('egitim',tr),('dogrulama',va),('test',te))},
        'pencere_ve_persistence': pt.to_dict('records')}
json.dump(prot, open('/content/drive/MyDrive/RuzgarTahmin/yeni/protokol_t57.json','w'), indent=1, default=str)
print('✓ Kaydedildi')

✓ Kaydedildi


In [13]:
!pip install catboost -q
import tensorflow as tf, pickle, time
from catboost import CatBoostRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
SEED = 42

def get_xy(L, H):
    out = {}
    for ad, x in (('tr',tr),('va',va),('te',te)):
        Xw, t, y, yc = make_windows(sc(x), x['Patv'].values, L, H)
        out[ad] = dict(X=Xw, t=t, y=y, yc=yc)
    return out

def build_lstm(L, F):
    m = Sequential([Input(shape=(L, F)),
                    LSTM(64, return_sequences=True), Dropout(0.2),
                    LSTM(32), Dropout(0.2), Dense(1)])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return m

def fit_lstm(D, L):
    tf.keras.utils.set_random_seed(SEED)
    m = build_lstm(L, D['tr']['X'].shape[2])
    m.fit(D['tr']['X'], D['tr']['y']/KAPASITE,
          validation_data=(D['va']['X'], D['va']['y']/KAPASITE),
          epochs=50, batch_size=256, verbose=0,
          callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
    return m

def pred_lstm(m, X):
    return np.clip(m.predict(X, verbose=0).flatten()*KAPASITE, 0, None)

# Geçmiş pencere seçimi: Raw LSTM, ufuk=1 saat (H=2), SADECE doğrulama MAE'sine göre
sec = {}
for L in SEQ_OPTS:
    t0 = time.time(); D = get_xy(L, 2); m = fit_lstm(D, L)
    p = pred_lstm(m, D['va']['X']); sec[L] = np.mean(np.abs(D['va']['y']-p))
    print(f"L={L} ({L/2} sa): doğrulama MAE={sec[L]:.2f}  | süre={time.time()-t0:.0f} sn")
L_BEST = min(sec, key=sec.get)
print('\nSeçilen geçmiş pencere: L =', L_BEST, f'({L_BEST/2} saat)')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00
L=4 (2.0 sa): doğrulama MAE=163.96  | süre=89 sn
L=8 (4.0 sa): doğrulama MAE=169.07  | süre=62 sn
L=12 (6.0 sa): doğrulama MAE=169.81  | süre=153 sn

Seçilen geçmiş pencere: L = 4 (2.0 saat)


In [14]:
results = {}
rows = []
for H in H_LIST:
    D = get_xy(L_BEST, H); te_ = D['te']
    # B1 Persistence
    results[('B1',H)] = dict(t=te_['t'], y=te_['y'], yc=te_['yc'], p=te_['yc'])
    rows.append(dict(model='B1 Persistence', H_saat=H/2, **metrics(te_['y'], te_['yc'], te_['yc'])))
    # B2 Raw LSTM
    m = fit_lstm(D, L_BEST); p2 = pred_lstm(m, te_['X'])
    results[('B2',H)] = dict(t=te_['t'], y=te_['y'], yc=te_['yc'], p=p2)
    rows.append(dict(model='B2 Raw LSTM', H_saat=H/2, **metrics(te_['y'], p2, te_['yc'])))
    # B3 CatBoost Direct (yalnız tahmin anındaki satır: son zaman adımı)
    cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, loss_function='MAE',
                           random_seed=SEED, verbose=0, early_stopping_rounds=50)
    cb.fit(D['tr']['X'][:,-1,:], D['tr']['y'], eval_set=(D['va']['X'][:,-1,:], D['va']['y']))
    p3 = np.clip(cb.predict(te_['X'][:,-1,:]), 0, None)
    results[('B3',H)] = dict(t=te_['t'], y=te_['y'], yc=te_['yc'], p=p3)
    rows.append(dict(model='B3 CatBoost Direct', H_saat=H/2, **metrics(te_['y'], p3, te_['yc'])))
    print(f'H={H/2} sa tamam')

tab = pd.DataFrame(rows)
print(tab.round(3).to_string(index=False))
tab.to_csv('/content/drive/MyDrive/RuzgarTahmin/yeni/adim2_T57_metrikler.csv', index=False)
pickle.dump(results, open('/content/drive/MyDrive/RuzgarTahmin/yeni/adim2_T57_tahminler.pkl','wb'))
print('✓ Kaydedildi')

H=0.5 sa tamam
H=1.0 sa tamam
H=2.0 sa tamam
H=3.0 sa tamam
H=4.0 sa tamam
H=6.0 sa tamam
             model  H_saat     MAE    RMSE   nMAE  nRMSE    R2   Skill
    B1 Persistence     0.5  77.799 145.738  5.187  9.716 0.874   0.000
       B2 Raw LSTM     0.5  90.041 141.989  6.003  9.466 0.880 -15.736
B3 CatBoost Direct     0.5  78.782 142.519  5.252  9.501 0.879  -1.263
    B1 Persistence     1.0 108.415 190.207  7.228 12.680 0.785   0.000
       B2 Raw LSTM     1.0 114.550 181.006  7.637 12.067 0.805  -5.659
B3 CatBoost Direct     1.0 106.116 179.891  7.074 11.993 0.808   2.120
    B1 Persistence     2.0 153.783 251.386 10.252 16.759 0.623   0.000
       B2 Raw LSTM     2.0 164.278 233.278 10.952 15.552 0.675  -6.824
B3 CatBoost Direct     2.0 143.355 228.520  9.557 15.235 0.689   6.781
    B1 Persistence     3.0 187.894 294.930 12.526 19.662 0.480   0.000
       B2 Raw LSTM     3.0 211.912 276.299 14.127 18.420 0.544 -12.783
B3 CatBoost Direct     3.0 168.054 260.040 11.204 17.336 0

In [15]:
def fit_lstm_t(D, L, delta):
    tf.keras.utils.set_random_seed(SEED)
    m = build_lstm(L, D['tr']['X'].shape[2])
    ytr = (D['tr']['y'] - (D['tr']['yc'] if delta else 0))/KAPASITE
    yva = (D['va']['y'] - (D['va']['yc'] if delta else 0))/KAPASITE
    m.fit(D['tr']['X'], ytr, validation_data=(D['va']['X'], yva), epochs=50, batch_size=256,
          verbose=0, callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
    return m

def pred_t(m, D, split, delta):
    p = m.predict(D[split]['X'], verbose=0).flatten()*KAPASITE
    if delta: p = p + D[split]['yc']
    return np.clip(p, 0, None)

for H in (2, 12):                      # 1 saat ve 6 saat
    D = get_xy(L_BEST, H)
    pers = np.mean(np.abs(D['va']['y']-D['va']['yc']))
    print(f"\nH={H/2} sa | doğrulama persistence MAE = {pers:.2f}")
    for delta in (False, True):
        m = fit_lstm_t(D, L_BEST, delta); p = pred_t(m, D, 'va', delta)
        print(f"  LSTM   hedef={'fark' if delta else 'seviye'}: doğrulama MAE = {np.mean(np.abs(D['va']['y']-p)):.2f}")
    for delta in (False, True):
        off_tr = D['tr']['yc'] if delta else 0; off_va = D['va']['yc'] if delta else 0
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, loss_function='MAE',
                               random_seed=SEED, verbose=0, early_stopping_rounds=50)
        cb.fit(D['tr']['X'][:,-1,:], D['tr']['y']-off_tr, eval_set=(D['va']['X'][:,-1,:], D['va']['y']-off_va))
        p = np.clip(cb.predict(D['va']['X'][:,-1,:]) + off_va, 0, None)
        print(f"  CatBst hedef={'fark' if delta else 'seviye'}: doğrulama MAE = {np.mean(np.abs(D['va']['y']-p)):.2f}")


H=1.0 sa | doğrulama persistence MAE = 165.32
  LSTM   hedef=seviye: doğrulama MAE = 163.96
  LSTM   hedef=fark: doğrulama MAE = 164.21
  CatBst hedef=seviye: doğrulama MAE = 155.81
  CatBst hedef=fark: doğrulama MAE = 156.52

H=6.0 sa | doğrulama persistence MAE = 340.16
  LSTM   hedef=seviye: doğrulama MAE = 292.87
  LSTM   hedef=fark: doğrulama MAE = 290.67
  CatBst hedef=seviye: doğrulama MAE = 282.87
  CatBst hedef=fark: doğrulama MAE = 285.70


In [16]:
!pip install vmdpy -q
from vmdpy import VMD
from joblib import Parallel, delayed
import os, time

ALPHA, TAU, K, DC, INIT, TOL = 2000, 0.0, 4, 0, 1, 1e-7
W = 672                                   # 14 gün (30 dk adımda)
OUT = '/content/drive/MyDrive/RuzgarTahmin/yeni/'

x_dec = d['Patv'].ffill().fillna(0).values.astype(float)    # nedensel doldurma
N = len(x_dec)

def vmd_sorted(sig):
    u, _, om = VMD(sig, ALPHA, TAU, K, DC, INIT, TOL)
    return u[np.argsort(om[-1])]           # modlar merkez frekansına göre sıralı (K, len)

def last_mode(sig):
    return vmd_sorted(sig)[:, -1]          # pencere sonundaki değer

# A) Full-sample (SIZDIRAN): tüm seriye tek VMD
t0 = time.time()
uA = vmd_sorted(x_dec).T
if uA.shape[0] < N: uA = np.vstack([uA, np.repeat(uA[-1:], N-uA.shape[0], 0)])
print(f'A) Full VMD tamam: {uA.shape} | {time.time()-t0:.0f} sn')

# B) Rolling (NEDENSEL): her τ için yalnızca [τ-W+1, τ]
uB = np.full((N, K), np.nan)
BLOCK = 2000
for s in range(W-1, N, BLOCK):
    e = min(s+BLOCK, N); fn = OUT + f'rollvmd_blk_{s}.npy'
    if os.path.exists(fn):
        uB[s:e] = np.load(fn); continue
    t0 = time.time()
    res = Parallel(n_jobs=-1)(delayed(last_mode)(x_dec[t-W+1:t+1]) for t in range(s, e))
    blk = np.nan_to_num(np.array(res)); np.save(fn, blk); uB[s:e] = blk
    print(f'B) blok {s}-{e} / {N} | {time.time()-t0:.0f} sn')

np.save(OUT+'vmd_full_T57.npy', uA); np.save(OUT+'vmd_roll_T57.npy', uB)
print('✓ Bileşenler kaydedildi')

A) Full VMD tamam: (35089, 4) | 26 sn
B) blok 671-2671 / 35089 | 175 sn
B) blok 2671-4671 / 35089 | 219 sn
B) blok 4671-6671 / 35089 | 114 sn
B) blok 6671-8671 / 35089 | 206 sn
B) blok 8671-10671 / 35089 | 242 sn
B) blok 10671-12671 / 35089 | 222 sn
B) blok 12671-14671 / 35089 | 172 sn
B) blok 14671-16671 / 35089 | 305 sn
B) blok 16671-18671 / 35089 | 263 sn
B) blok 18671-20671 / 35089 | 181 sn
B) blok 20671-22671 / 35089 | 250 sn
B) blok 22671-24671 / 35089 | 235 sn
B) blok 24671-26671 / 35089 | 219 sn
B) blok 26671-28671 / 35089 | 241 sn
B) blok 28671-30671 / 35089 | 269 sn
B) blok 30671-32671 / 35089 | 293 sn
B) blok 32671-34671 / 35089 | 222 sn
B) blok 34671-35089 / 35089 | 52 sn
✓ Bileşenler kaydedildi


In [17]:
tau = 20000
rng = np.random.default_rng(0)
x2 = x_dec.copy(); x2[tau+1:] += rng.normal(0, 200, N-tau-1)      # GELECEĞİ boz

# B (rolling): τ anındaki bileşen, gelecek bozulunca değişmemeli
b_orig = last_mode(x_dec[tau-W+1:tau+1]); b_pert = last_mode(x2[tau-W+1:tau+1])
print('B  bileşen farkı (gelecek bozulunca):', np.abs(b_orig-b_pert).max(), '  [beklenen: 0]')
print('B  kayıtlı değer = taze hesap?      :', np.allclose(uB[tau], b_orig))

# A (full-sample): τ anındaki bileşen, gelecek bozulunca DEĞİŞİR → sızıntı
a_pert = vmd_sorted(x2).T[tau]
print('A  bileşen farkı (gelecek bozulunca):', np.abs(uA[tau]-a_pert).max(), '  [beklenen: >0 → sızıntı]')

B  bileşen farkı (gelecek bozulunca): 0.0   [beklenen: 0]
B  kayıtlı değer = taze hesap?      : True
A  bileşen farkı (gelecek bozulunca): 41.071581929117784   [beklenen: >0 → sızıntı]


In [18]:
def scale_comp(U):
    pos = d.index.get_indexer(tr.index)
    lo, hi = np.nanmin(U[pos],0), np.nanmax(U[pos],0)
    return (U-lo)/np.where(hi-lo==0, 1, hi-lo)

uA_m = uA.copy(); uA_m[:W-1] = np.nan        # B ile aynı örnek kümesi (ilk 14 gün her ikisinde de yok)
Us = {'B6': scale_comp(uA_m), 'B7': scale_comp(uB)}

def get_xy_ext(L, H, U):
    out = {}
    for ad, x in (('tr',tr),('va',va),('te',te)):
        idx = d.index.get_indexer(x.index)
        X = np.hstack([sc(x), U[idx]])
        Xw, t, y, yc = make_windows(X, x['Patv'].values, L, H)
        out[ad] = dict(X=Xw, t=t, y=y, yc=yc)
    return out

rows = []
for H in H_LIST:
    for ad_, ad_ad in (('B6','B6 Full VMD–LSTM (sızdıran)'), ('B7','B7 Rolling VMD–LSTM (nedensel)')):
        D = get_xy_ext(L_BEST, H, Us[ad_]); te_ = D['te']
        assert np.array_equal(te_['t'], results[('B1',H)]['t']), 'test örnek kümesi farklı!'
        m = fit_lstm(D, L_BEST); p = pred_lstm(m, te_['X'])
        results[(ad_,H)] = dict(t=te_['t'], y=te_['y'], yc=te_['yc'], p=p)
        rows.append(dict(model=ad_ad, H_saat=H/2, **metrics(te_['y'], p, te_['yc'])))
    print(f'H={H/2} sa tamam')

tab2 = pd.DataFrame(rows)
print(tab2.round(3).to_string(index=False))
# ön LSI: aynı model ailesi (VMD-LSTM), leaky vs causal
print('\nÖn LSI (VMD–LSTM, B6 sızdıran vs B7 nedensel):')
for H in H_LIST:
    ml = np.mean(np.abs(results[('B6',H)]['y']-results[('B6',H)]['p']))
    mc = np.mean(np.abs(results[('B7',H)]['y']-results[('B7',H)]['p']))
    print(f'  H={H/2} sa: MAE leaky={ml:.2f}  causal={mc:.2f}  LSI={lsi(mc,ml):.2f}%')
tab2.to_csv(OUT+'adim3_T57_metrikler.csv', index=False)
pickle.dump(results, open(OUT+'adim2_T57_tahminler.pkl','wb'))
print('✓ Kaydedildi')

H=0.5 sa tamam
H=1.0 sa tamam
H=2.0 sa tamam
H=3.0 sa tamam
H=4.0 sa tamam
H=6.0 sa tamam
                         model  H_saat     MAE    RMSE   nMAE  nRMSE    R2   Skill
   B6 Full VMD–LSTM (sızdıran)     0.5  53.294  82.771  3.553  5.518 0.959  31.497
B7 Rolling VMD–LSTM (nedensel)     0.5  86.764 141.518  5.784  9.435 0.881 -11.523
   B6 Full VMD–LSTM (sızdıran)     1.0  64.713 101.664  4.314  6.778 0.939  40.309
B7 Rolling VMD–LSTM (nedensel)     1.0 116.107 181.219  7.740 12.081 0.805  -7.095
   B6 Full VMD–LSTM (sızdıran)     2.0  67.732 100.465  4.515  6.698 0.940  55.956
B7 Rolling VMD–LSTM (nedensel)     2.0 158.975 231.230 10.598 15.415 0.681  -3.376
   B6 Full VMD–LSTM (sızdıran)     3.0  73.429 110.986  4.895  7.399 0.926  60.920
B7 Rolling VMD–LSTM (nedensel)     3.0 191.004 266.647 12.734 17.776 0.575  -1.655
   B6 Full VMD–LSTM (sızdıran)     4.0  96.808 138.493  6.454  9.233 0.885  54.969
B7 Rolling VMD–LSTM (nedensel)     4.0 215.858 294.953 14.391 19.664 0.478  -0.4

In [19]:
from scipy.signal import butter, lfilter, lfilter_zi, filtfilt

# 1) Bantları YALNIZCA eğitim bölümünden öğren ve kilitle
x_tr_seg = x_dec[:len(tr)]
_, _, om_tr = VMD(x_tr_seg, ALPHA, TAU, K, DC, INIT, TOL)
fc = np.sort(om_tr[-1])
print('Eğitimden öğrenilen merkez frekanslar (döngü/adım):', fc.round(5))
fcl = np.clip(fc, 1e-4, 0.49)
edges = np.sqrt(fcl[:-1]*fcl[1:])                 # K-1 sınır (geometrik orta)
print('Bant sınırları:', edges.round(5))

bands = []
for k in range(K):
    if k == 0:       ba = butter(3, edges[0], 'low', fs=1.0)
    elif k == K-1:   ba = butter(3, edges[K-2], 'high', fs=1.0)
    else:            ba = butter(3, [edges[k-1], edges[k]], 'band', fs=1.0)
    bands.append(ba)

def make_lcsf(x, causal=True):
    out = np.zeros((len(x), K))
    for k, (b, a) in enumerate(bands):
        if causal:
            zi = lfilter_zi(b, a)*x[0]; out[:,k] = lfilter(b, a, x, zi=zi)[0]
        else:
            out[:,k] = filtfilt(b, a, x)           # çift yönlü → geleceğe bakar (SIZDIRAN)
    return out

cC = make_lcsf(x_dec, True); cL = make_lcsf(x_dec, False)
cC[:W-1] = np.nan; cL[:W-1] = np.nan              # B6/B7 ile aynı örnek kümesi
Us['B8']  = scale_comp(cC); Us['B8L'] = scale_comp(cL)

# nedensellik kanıtı
x2 = x_dec.copy(); x2[20001:] += np.random.default_rng(0).normal(0, 200, N-20001)
print('Nedensel LCSF fark (gelecek bozulunca):', np.abs(make_lcsf(x_dec,True)[20000]-make_lcsf(x2,True)[20000]).max(), '[beklenen 0]')
print('Çift yönlü LCSF fark                  :', np.abs(make_lcsf(x_dec,False)[20000]-make_lcsf(x2,False)[20000]).max().round(2), '[beklenen >0 → sızıntı]')

Eğitimden öğrenilen merkez frekanslar (döngü/adım): [0.00054 0.01849 0.04388 0.09026]
Bant sınırları: [0.00316 0.02848 0.06293]
Nedensel LCSF fark (gelecek bozulunca): 0.0 [beklenen 0]
Çift yönlü LCSF fark                  : 33.79 [beklenen >0 → sızıntı]
